# <span style="color:#e0bda8">1. Import Packages and Libraries</span>

In [35]:
# 1. Data Manipulation
# =====================================================
import pandas as pd
import numpy as np


# 2. Dimensionality Reduction
# =====================================================
from sklearn.decomposition import PCA


# 3. Data Visualization
# =====================================================
import matplotlib.pyplot as plt
import seaborn as sns


# 4. Statistical Analysis
# =====================================================
from scipy.stats import kendalltau


# 5. Path Configuration
# =====================================================
import os


# 6. Utilities
# =====================================================
import warnings

warnings.filterwarnings('ignore')

# <span style="color:#e0bda8">2. Project Structure and Directory Configuration </span>   


In [36]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA = os.path.join(PROJECT_ROOT, "01_Data")
DATA_RAW = os.path.join(DATA, "01_Raw")
DATA_PROCESSED = os.path.join(DATA, "02_Processed")
DATA_STATS = os.path.join(DATA, "03_Statistics")
DATA_INDEX = os.path.join(DATA, "04_Index_Final")

CLUSTER_PCA = os.path.join(PROJECT_ROOT, "03_Clustering_PCA")
CLUSTER_PCA_EXCEL = os.path.join(CLUSTER_PCA, "01_Excel")
CLUSTER_PCA_FIG = os.path.join(CLUSTER_PCA, "02_Figures")

METHOD = os.path.join(PROJECT_ROOT, "04_Method_Comparison")
METHOD_EXCEL = os.path.join(METHOD, "01_Excel")
METHOD_FIG = os.path.join(METHOD, "02_Figures")

ML_RESULTS = os.path.join(PROJECT_ROOT, "05_ML_Results")
ML_METRICS = os.path.join(ML_RESULTS, "01_Metrics")
ML_PRED = os.path.join(ML_RESULTS, "02_Predictions")
ML_MODELS = os.path.join(ML_RESULTS, "03_Models")
ML_SHAP = os.path.join(ML_RESULTS, "04_SHAP")
ML_SHAP_EXCEL = os.path.join(ML_SHAP, "01_Excel")
ML_SHAP_FIG = os.path.join(ML_SHAP, "02_Figures")

# <span style="color:#e0bda8">3. Reading the Data</span>

In [37]:
df_env_score = pd.read_csv(os.path.join(DATA_PROCESSED, 'df_env_score.csv')) 
df_env_score = df_env_score.set_index(['Economy', 'Year'])

df_soc_score = pd.read_csv(os.path.join(DATA_PROCESSED, 'df_soc_score.csv'))
df_soc_score = df_soc_score.set_index(['Economy', 'Year'])

df_gov_score = pd.read_csv(os.path.join(DATA_PROCESSED, 'df_gov_score.csv'))
df_gov_score = df_gov_score.set_index(['Economy', 'Year'])

# <span style="color:#e0bda8">4. Index Scores Construction </span>

## <span style="color:#e0bda8">4.1. Equal Weights </span>

In [38]:
def equal_weighted_index(df):

    # 1️⃣ Cria um array de pesos iguais
    n_vars = df.shape[1]
    weights = np.ones(n_vars) / n_vars  # soma 1

    # 2️⃣ Calcula score ponderado
    score = df.values @ weights

    # 3️⃣ Score máximo teórico baseado nos máximos reais das colunas
    col_max = df.max(axis=0).values  # máximo de cada coluna
    score_max = np.sum(weights * col_max)  # soma ponderada dos máximos

    col_min = df.min(axis=0).values
    score_min = np.sum(weights * col_min)

    # 3️⃣ Converte para 0–100
    score_0_100 =  100 * (score - score_min) / (score_max - score_min)
    # 4️⃣ Retorna DataFrame com scores
    score_df = pd.DataFrame(score_0_100, index=df.index, columns=['Score'])
    
    return score_df

In [39]:
E_score_equal = equal_weighted_index(df_env_score)
S_score_equal = equal_weighted_index(df_soc_score)
G_score_equal = equal_weighted_index(df_gov_score)

## <span style="color:#e0bda8">4.2. PCA Explained Variance Weights </span>

In [40]:
def pca_variance_weighted_index(df, n_components=3):

    # 1️⃣ Ajusta PCA
    pca = PCA(n_components=n_components)
    pca.fit(df)

    # 2️⃣ Calcula loadings absolutos
    loadings = np.abs(pca.components_.T)  # indicadores × componentes

    # 3️⃣ Pega variância explicada pelos componentes
    explained_var = pca.explained_variance_ratio_

    # 4️⃣ Calcula pesos ponderados (loadings × variância explicada)
    raw_weights = loadings @ explained_var

    # 5️⃣ Normaliza para que somem 1
    weights = raw_weights / raw_weights.sum()

    # 6️⃣ Calcula score para cada país
    score = df.values @ weights

    # 3️⃣ Score máximo teórico baseado nos máximos reais das colunas
    col_max = df.max(axis=0).values  # máximo de cada coluna
    score_max = np.sum(weights * col_max)  # soma ponderada dos máximos

    col_min = df.min(axis=0).values
    score_min = np.sum(weights * col_min)

    # 4️⃣ Normaliza para 0–100
    score_0_100 = 100 * (score - score_min) / (score_max - score_min)

    # 5️⃣ Retorna DataFrame com scores
    score_df = pd.DataFrame(score_0_100, index=df.index, columns=['Score'])

    return score_df

In [41]:
E_score_pca = pca_variance_weighted_index(df_env_score) 
S_score_pca = pca_variance_weighted_index(df_soc_score) 
G_score_pca = pca_variance_weighted_index(df_gov_score) 

## <span style="color:#e0bda8">4.3. Variance-Based Weights </span>

In [42]:
def variance_weighted_index(df):
    
    # 1️⃣ Calcula a variância de cada coluna
    variances = df.var(axis=0)
    
    # 2️⃣ Normaliza para somar 1
    weights = variances / variances.sum()
    
    # 3️⃣ Calcula score ponderado
    score = df.values @ weights
    
    # 3️⃣ Score máximo teórico baseado nos máximos reais das colunas
    col_max = df.max(axis=0).values  # máximo de cada coluna
    score_max = np.sum(weights * col_max)  # soma ponderada dos máximos

    col_min = df.min(axis=0).values
    score_min = np.sum(weights * col_min)

    # 3️⃣ Converte para 0–100
    score_0_100 =  100 * (score - score_min) / (score_max - score_min)
    
    # 6️⃣ Retorna DataFrame com scores
    score_df = pd.DataFrame(score_0_100, index=df.index, columns=['Score'])
    
    return score_df

In [43]:
E_score_variance = variance_weighted_index(df_env_score)
S_score_variance = variance_weighted_index(df_soc_score)
G_score_variance = variance_weighted_index(df_gov_score)

## <span style="color:#e0bda8">4.4. Entropy-Based Weights </span>

In [44]:
def entropy_weighted_index(df, epsilon=1e-12):
    
    # 1️⃣ Normaliza cada coluna para somar 1 (distribuição de probabilidade)
    p = df.div(df.sum(axis=0) + epsilon, axis=1)
    
    # 2️⃣ Calcula entropia de cada variável
    k = 1 / np.log(df.shape[0])  # constante de normalização
    entropy = -k * (p * np.log(p + epsilon)).sum(axis=0)
    
    # 3️⃣ Calcula a "informação" (quanto menor a entropia, mais informativa)
    info = 1 - entropy
    
    # 4️⃣ Calcula pesos normalizados
    weights = info / info.sum()
    
    # 5️⃣ Calcula score ponderado
    score = df.values @ weights.values
    
    # 3️⃣ Score máximo teórico baseado nos máximos reais das colunas
    col_max = df.max(axis=0).values  # máximo de cada coluna
    score_max = np.sum(weights * col_max)  # soma ponderada dos máximos

    col_min = df.min(axis=0).values
    score_min = np.sum(weights * col_min)

    # 3️⃣ Converte para 0–100
    score_0_100 =  100 * (score - score_min) / (score_max - score_min)
    
    # 7️⃣ Retorna DataFrame com scores
    score_df = pd.DataFrame(score_0_100, index=df.index, columns=['Score'])
    
    return score_df

In [45]:
E_score_entropy = entropy_weighted_index(df_env_score)
S_score_entropy = entropy_weighted_index(df_soc_score)
G_score_entropy = entropy_weighted_index(df_gov_score)

# <span style="color:#e0bda8">5. Comparison of Score Weighting Methods </span>

## <span style="color:#e0bda8">5.1. Score Aggregation by Pillar </span>

In [46]:
E_scores = [E_score_equal, E_score_pca, E_score_entropy, E_score_variance]
S_scores = [S_score_equal, S_score_pca, S_score_entropy, S_score_variance]
G_scores = [G_score_equal, G_score_pca, G_score_entropy, G_score_variance]

In [47]:
# Flatten any returned DataFrames with a single 'Score' column into Series
for name in ['E_score_equal','E_score_pca','E_score_entropy','E_score_variance']:
    obj = globals().get(name)
    if isinstance(obj, pd.DataFrame) and 'Score' in obj.columns:
        globals()[name] = obj['Score']

for name in ['S_score_equal','S_score_pca','S_score_entropy','S_score_variance']:
    obj = globals().get(name)
    if isinstance(obj, pd.DataFrame) and 'Score' in obj.columns:
        globals()[name] = obj['Score']

for name in ['G_score_equal','G_score_pca','G_score_entropy','G_score_variance']:
    obj = globals().get(name)
    if isinstance(obj, pd.DataFrame) and 'Score' in obj.columns:
        globals()[name] = obj['Score']

# Rebuild score lists (in case they were overwritten earlier)
E_scores = [E_score_equal, E_score_pca, E_score_entropy, E_score_variance]
S_scores = [S_score_equal, S_score_pca, S_score_entropy, S_score_variance]
G_scores = [G_score_equal, G_score_pca, G_score_entropy, G_score_variance]

# Concatenate series into DataFrames with single-level columns (methods)
E_combined = pd.concat(E_scores, axis=1, keys=['Equal', 'PCA', 'Entropy', 'Variance'])
S_combined = pd.concat(S_scores, axis=1, keys=['Equal', 'PCA', 'Entropy', 'Variance'])
G_combined = pd.concat(G_scores, axis=1, keys=['Equal', 'PCA', 'Entropy', 'Variance'])

## <span style="color:#e0bda8">5.2. Evaluation of Score Methods</span>

In [48]:
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

class ESGScoreEvaluator:

    def __init__(self, E_combined, S_combined, G_combined):
        self.E_combined = E_combined
        self.S_combined = S_combined
        self.G_combined = G_combined

        self.pillars = {
            'Environmental': self.E_combined,
            'Social': self.S_combined,
            'Governance': self.G_combined
        }

        self.methods = ['Equal', 'PCA', 'Entropy', 'Variance']
        self.results = {}

    def evaluate_internal_consistency(self):

        print("=" * 80)
        print("CRITERION 1: INTERNAL CONSISTENCY")
        print("=" * 80)

        consistency_results = {}

        for pillar_name, df in self.pillars.items():
            print(f"\n{pillar_name} Pillar:")
            print("-" * 60)

            corr_matrix = df.corr(method='pearson')

            mask = np.triu(np.ones_like(corr_matrix), k=1).astype(bool)
            avg_correlation = corr_matrix.where(mask).stack().mean()
            pca = PCA(n_components=1)
            pca.fit(df.dropna())
            variance_explained = pca.explained_variance_ratio_[0]

            cv_scores = (df.std(axis=1)/df.mean(axis=1)).mean()

            consistency_results[pillar_name] = {
                'correlation_matrix': corr_matrix,
                'avg_correlation': avg_correlation,
                'variance_explained_pc1': variance_explained,
                'mean_cv': cv_scores
            }
            
            print(f'Average Pairwise Correlation: {avg_correlation:.4f}')
            print(f'Variance Explained by PC1: {variance_explained:.4f}')
            print(f'Mean Coefficient of Variation: {cv_scores:.4f}')
            print(f'\n Correlation Matrix:')
            print(corr_matrix.round(4))

        self.results['internal_consistency'] = consistency_results
        return consistency_results
        
    def evaluate_robustnesss_to_extremes(self):

        print("\n" + "=" * 80)
        print("CRITERION 2: ROBUSTNESS TO EXTREMES")
        print("=" * 80)

        robustness_results = {}

        for pillar_name, df in self.pillars.items():
            print(f"\n{pillar_name} Pillar:")
            print("-" * 60)

            method_robustness = {}

            for method in self.methods:
                scores = df[method].dropna()

                # Identify outliers using IQR method
                Q1 = scores.quantile(0.25)
                Q3 = scores.quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR

                outliers = scores[(scores < lower_bound) | (scores > upper_bound)]
                outlier_pct = len(outliers) / len(scores) * 100

                # Skewness and kurtosis
                skewness = scores.skew()
                kurtosis = scores.kurtosis()

                # Range and interquartile range
                score_range = scores.max() - scores.min()
                iqr = IQR

                # Robust coefficient of variation (using median and MAD)
                mad = np.median(np.abs(scores - scores.median()))
                robust_cv = mad / scores.median() if scores.median() != 0 else np.nan

                method_robustness[method] = {
                    'outlier_percentage': outlier_pct,
                    'skewness': skewness,
                    'kurtosis': kurtosis,
                    'range': score_range,
                    'iqr': iqr,
                    'robust_cv': robust_cv
                }

                print(f"\n {method} Method:")
                print(f" Outlier Percentage: {outlier_pct:.2f}%")
                print(f" Skewness: {skewness:.4f}")
                print(f" Kurtosis: {kurtosis:.4f}")
                print(f" Robust CV: {robust_cv:.4f}")

            robustness_results[pillar_name] = method_robustness
            
        self.results['robustness'] = robustness_results
        return robustness_results
    
    def evaluate_ranking_stability(self): 

        print("\n" + "=" * 80)
        print("CRITERION 3: RANKING STABILITY")
        print("=" * 80)

        stability_results = {}

        for pillar_name, df in self.pillars.items():
            print(f"\n{pillar_name} Pillar:")
            print("-" * 60)

            rankings = df.rank(ascending=False, method='average')

            rank_corr_matrix = rankings.corr(method='spearman')
            avg_rank_corr = rank_corr_matrix.where(np.triu(np.ones_like(rank_corr_matrix), k=1).astype(bool)).stack().mean()

            kendall_df = pd.DataFrame(index=self.methods, columns=self.methods, dtype=float)
            kendall_scores = {}

            for i, method1 in enumerate(self.methods):
                for method2 in self.methods[i+1:]:
                    valid_idx = rankings[[method1, method2]].dropna().index
                    if len(valid_idx) > 0:
                        tau, _ = kendalltau(rankings.loc[valid_idx, method1], rankings.loc[valid_idx, method2])
                        kendall_scores[f"{method1} vs {method2}"] = tau
                        kendall_df.loc[method1, method2] = tau
                        kendall_df.loc[method2, method1] = tau 
            
            np.fill_diagonal(kendall_df.values, 1)
            avg_kendall = np.mean(list(kendall_scores.values()))

            rank_displacements_df = pd.DataFrame(index=self.methods, columns=self.methods, dtype=float)

            for i, method1 in enumerate(self.methods):
                displacements = []
                for method2 in self.methods[i+1:]:
                    valid_idx = rankings[[method1, method2]].dropna().index
                    if len(valid_idx) > 0:
                        displacement = np.abs(rankings.loc[valid_idx, method1] - rankings.loc[valid_idx, method2]).mean()
                        rank_displacements_df.loc[method1, method2] = displacement
                        rank_displacements_df.loc[method2, method1] = displacement

            np.fill_diagonal(rank_displacements_df.values, 0)
            rank_displacements = rank_displacements_df.where(~np.eye(len(rank_displacements_df),dtype=bool)).mean(axis=1).to_dict()

            top10_overlaps_df = pd.DataFrame(index=self.methods, columns=self.methods, dtype=float)
            top10_overlaps = {}

            for i, method1 in enumerate(self.methods):
                overlaps = []
                top10_method1 = set(rankings[method1].nsmallest(10).index)
                for method2 in self.methods[i+1:]:
                    top10_method2 = set(rankings[method2].nsmallest(10).index)
                    overlap = len(top10_method1.intersection(top10_method2)) / 10 * 100
                    top10_overlaps_df.loc[method1, method2] = overlap
                    top10_overlaps_df.loc[method2, method1] = overlap

            np.fill_diagonal(top10_overlaps_df.values, 100)
            top10_overlaps = top10_overlaps_df.where(~np.eye(len(top10_overlaps_df),dtype=bool)).mean(axis=1).to_dict()


            stability_results[pillar_name] = {
                'rank_correlation_matrix': rank_corr_matrix,
                'avg_spearman': avg_rank_corr,
                'avg_kendall': avg_kendall,
                'kendall_pairwise': kendall_df,   
                'rank_displacements': rank_displacements,
                'rank_displacements_pairwise': rank_displacements_df,
                'top10_overlaps': top10_overlaps,
                'top10_overlaps_pairwise': top10_overlaps_df
            }


            print(f'Average Spearman Correlation: {avg_rank_corr:.4f}')
            print(f'Average Kendall Tau: {avg_kendall:.4f}')
            print(f'Average Rank Displacements by Method:')
            for method, disp in rank_displacements.items():
                print(f' {method}: {disp:.2f} positions') 
            print(f'Average Top-10 Overlap (%):')
            for method, overlap in top10_overlaps.items():
                print(f' {method}: {overlap:.1f}%')
            
        self.results['stability'] = stability_results
        return stability_results
    
    def evaluate_interpretability(self):
            
        print("\n" + "=" * 80)
        print("CRITERION 4: INTERPRETABILITY")
        print("=" * 80)

        interpretability_results = {}

        for pillar_name, df in self.pillars.items():
            print(f"\n{pillar_name} Pillar:")
            print("-" * 60)

            method_interpret = {}

            for method in self.methods: 
                scores = df[method].dropna()

                mean_score = scores.mean()
                median_score = scores.median()
                std_score = scores.std()

                actual_range = scores.max() - scores.min()
                range_utilization = actual_range / 100 * 100

                deciles = pd.qcut(scores, q=10, duplicates='drop', labels=False)
                decile_counts = pd.Series(deciles).value_counts(normalize=True)
                entropy = -np.sum(decile_counts * np.log(decile_counts + 1e-10))
                max_entropy = np.log(len(decile_counts))
                normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0 

                sorted_scores = np.sort(scores)
                n = len(sorted_scores)
                index = np.arange(1, n + 1)
                gini = (2 * np.sum(index * sorted_scores)) / (n * np.sum(sorted_scores)) - (n + 1) / n

                method_interpret[method] = {
                    'mean': mean_score,
                    'median': median_score,
                    'std': std_score,
                    'range_utilization': range_utilization,
                    'normalized_entropy': normalized_entropy,
                    'gini_coefficient': gini
                }

                print(f"\n {method} Method:")
                print(f" Mean: {mean_score:.2f}")
                print(f" Median: {median_score:.2f}")
                print(f" Standard Deviation: {std_score:.2f}")
                print(f" Range Utilization: {range_utilization:.1f}%")
                print(f" Discriminatory Power (Entropy): {normalized_entropy:.4f}")
                print(f" Gini Coefficient: {gini:.4f}")

                interpretability_results[pillar_name] = method_interpret 

        self.results['interpretability'] = interpretability_results
        return interpretability_results
        
    def generate_comparative_summary(self):

        print("\n" + "=" * 80)
        print("COMPARATIVE SUMMARY: METHOD PERFORMANCE ACROSS CRITERIA")
        print("=" * 80)

        summary = {}

        for pillar_name, df in self.pillars.items():
            print(f"\n{pillar_name} Pillar:")
            print("-" * 60)

            method_scores = {method: [] for method in self.methods}

            if 'internal_consistency' in self.results:
                corr_matrix = self.results['internal_consistency'][pillar_name]['correlation_matrix']
                for method in self.methods:
                    other_methods = [m for m in self.methods if m != method]
                    avg_corr = corr_matrix.loc[method, other_methods].mean()
                    method_scores[method].append(avg_corr)
            
            if 'robustness' in self.results:
                for method in self.methods:
                    rob = self.results['robustness'][pillar_name][method]

                    outlier_score = 1 - (rob['outlier_percentage'] / 100)
                    skew_score = 1 - min(abs(rob['skewness']) / 2, 1)
                    kurt_score = 1 - min(abs(rob['kurtosis']) / 5, 1)
                    robustness_score = (outlier_score + skew_score + kurt_score) / 3

                    method_scores[method].append(robustness_score)

            if 'stability' in self.results:
                stab = self.results['stability'][pillar_name]
                for method in self.methods:
                    stability_score = stab['top10_overlaps'][method] / 100
            
                    method_scores[method].append(stability_score)
            
            if 'interpretability' in self.results:
                for method in self.methods:
                    interp = self.results['interpretability'][pillar_name][method]
                    interpret_score = (interp['range_utilization'] / 100) / 2

                    method_scores[method].append(interpret_score)

            overall_scores = {method: np.mean(scores) for method, scores in method_scores.items()}
            summary[pillar_name] = {
                'criterion_scores': method_scores,
                'overall_scores': overall_scores,
                'recommended_method': max(overall_scores, key=overall_scores.get)
            }

            print("\n Overall Performance Scores (0-1 scale)")
            for method in self.methods:
                print (f" {method}: {overall_scores[method]:.4f}")
            
            print(f"\n Recommended Method: {summary[pillar_name]['recommended_method']}")

        self.results['summary'] = summary
        return summary
    
    def create_visualizations(self):
            
        print("\n" + "=" * 80)
        print("GENERATING VISUALIZATIONS")
        print("=" * 80)

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for idx, (pillar_name, data) in enumerate(self.results['internal_consistency'].items()):
            sns.heatmap(data['correlation_matrix'], annot=True, fmt='.3f', cmap='RdYlGn', 
                        center=0.95, vmin=0.8, vmax=1.0, ax=axes[idx], cbar_kws={'label': 'Correlation'}
            )
            axes[idx].set_title(f'{pillar_name} Pillar\nInternal Consistency', fontsize=12, fontweight= 'bold')
        plt.tight_layout()

        plt.savefig(os.path.join(METHOD_FIG, "esg_internal_consistency.png"), dpi=300, bbox_inches="tight")
        plt.close()
        print(f" Saved: esg_internal_consistency.png")


        fig, axes = plt.subplots(3, 4, figsize=(20, 15))
        for row, (pillar_name, df) in enumerate(self.pillars.items()):
            for col, method in enumerate(self.methods):
                ax = axes[row, col]
                scores = df[method].dropna()
                ax.hist(scores, bins=30, alpha=0.7, edgecolor='black')
                ax.axvline(scores.mean(), color='red', linestyle='--', linewidth=2,
                           label=f'Mean: {scores.mean():.1f}')
                ax.axvline(scores.median(), color='blue', linestyle='--', linewidth=2,
                           label=f'Median: {scores.median():.1f}')
                ax.set_xlabel('Score', fontsize=10)
                ax.set_ylabel('Frequency', fontsize=10)
                ax.set_title(f'{pillar_name[0]}-Pillar: {method}', fontsize=11, fontweight='bold')
                ax.legend(fontsize=8)
                ax.grid(alpha=0.3)
        plt.tight_layout()

        plt.savefig(os.path.join(METHOD_FIG, "esg_score_distributions.png"), dpi=300, bbox_inches="tight")
        plt.close()
        print(f" Saved: esg_score_distributions.png")

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for idx, pillar_name in enumerate(self.pillars.keys()):
            rank_corr = self.results['stability'][pillar_name]['rank_correlation_matrix']
            sns.heatmap(rank_corr, annot=True, fmt='.3f', cmap='RdYlGn', center=0.95, 
                        vmin=0.8, vmax=1.0,
                        ax=axes[idx], cbar_kws={'label': 'Spearman p'}
            )

            axes[idx].set_title(f'{pillar_name} Pillar\nRanking Stability (Spearman)', fontsize=12, fontweight='bold')
        plt.tight_layout()

        plt.savefig(os.path.join(METHOD_FIG, "esg_spearman_pairwise.png"), dpi=300, bbox_inches="tight")
        plt.close()
        print(f" Saved: esg_spearman_pairwise.png")

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for idx, pillar_name in enumerate(self.pillars.keys()):
            rank_corr = self.results['stability'][pillar_name]['kendall_pairwise']
            sns.heatmap(rank_corr, annot=True, fmt='.3f', cmap='RdYlGn', center=0.95, 
                        vmin=0.5, vmax=1.0,
                        ax=axes[idx], cbar_kws={'label': 'Kendall Tau'}
            )

            axes[idx].set_title(f'{pillar_name} Pillar\nRanking Stability (Kendall Tau)', fontsize=12, fontweight='bold')
        plt.tight_layout()

        plt.savefig(os.path.join(METHOD_FIG, "esg_kendall_pairwise.png"), dpi=300, bbox_inches="tight")
        plt.close()
        print(f" Saved: esg_kendall_pairwise.png")

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for idx, pillar_name in enumerate(self.pillars.keys()):
            rank_corr = self.results['stability'][pillar_name]['rank_displacements_pairwise']
            sns.heatmap(rank_corr, annot=True, fmt='.3f', cmap='RdYlGn', center=0.95, 
                        vmin=0, vmax=50,
                        ax=axes[idx], cbar_kws={'label': 'Rank Displacement'}
            )

            axes[idx].set_title(f'{pillar_name} Pillar\nRanking Stability (Rank Displacement)', fontsize=12, fontweight='bold')
        plt.tight_layout()

        plt.savefig(os.path.join(METHOD_FIG, "esg_rank_displacements_pairwise.png"), dpi=300, bbox_inches="tight")
        plt.close()
        print(f" Saved: esg_rank_displacements_pairwise.png")


        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for idx, pillar_name in enumerate(self.pillars.keys()):
            rank_corr = self.results['stability'][pillar_name]['top10_overlaps_pairwise']
            sns.heatmap(rank_corr, annot=True, fmt='.3f', cmap='RdYlGn', center=0.95, 
                        vmin=0, vmax=100,
                        ax=axes[idx], cbar_kws={'label': 'Top 10 Overlap (%)'}
            )

            axes[idx].set_title(f'{pillar_name} Pillar\nRanking Stability (Top-10 Overlap)', fontsize=12, fontweight='bold')
        plt.tight_layout()

        plt.savefig(os.path.join(METHOD_FIG, "esg_top10_overlaps_pairwise.png"), dpi=300, bbox_inches="tight")
        plt.close()
        print(f" Saved: esg_top10_overlaps_pairwise.png")

        fig, axes = plt.subplots(1, 3, figsize=(18, 6), subplot_kw=dict(projection='polar'))
        criteria = ['Internal\nConsistency', 'Robustness', 'Stability', 'Interpretability']

        for idx, pillar_name in enumerate(self.pillars.keys()):
            ax = axes[idx]
            angles = np.linspace(0, 2 * np.pi, len(criteria), endpoint=False).tolist()
            angles += angles[:1]

            for method in self.methods:
                scores = self.results['summary'][pillar_name]['criterion_scores'][method]
                scores += scores[:1]

                ax.plot(angles, scores, 'o-', linewidth=2, label=method)
                ax.fill(angles, scores, alpha=0.15)
            
            ax.set_xticks(angles[:-1])
            ax.set_xticklabels(criteria, fontsize=9)
            ax.set_ylim(0,1)
            ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
            ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8)
            ax.set_title(f'{pillar_name} Pillar\nMethod Comparison', fontsize=12, 
                         fontweight='bold', pad=20)
            ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
            ax.grid(True)
        plt.tight_layout()

        plt.savefig(os.path.join(METHOD_FIG, "esg_method_comparison_radar.png"), dpi=300, bbox_inches="tight")
        plt.close()
        print(f" Saved: esg_method_comparison_radar.png")

        fig, ax = plt.subplots(figsize=(12, 6))
        pillar_names = list(self.pillars.keys())
        x = np.arange(len(pillar_names))
        width = 0.2

        for idx, method in enumerate(self.methods):
            scores = [self.results['summary'][pillar]['overall_scores'][method] for pillar in pillar_names]
            ax.bar(x + idx * width, scores, width, label=method)
        ax.set_xlabel('ESG Pillar', fontsize=12, fontweight='bold')
        ax.set_ylabel('Overall Performance Score', fontsize=12, fontweight='bold')
        ax.set_title('Comparative Performance of Aggregation Methods Across ESG Pillars', 
                     fontsize=14, fontweight='bold')
        ax.set_xticks(x + width * 1.5)
        ax.set_xticklabels(pillar_names)
        ax.legend(title='Method', fontsize=10)
        ax.grid(axis='y', alpha=0.3)
        ax.set_ylim(0, 1.1)
        plt.tight_layout()

        plt.savefig(os.path.join(METHOD_FIG, "esg_overall_performance.png"), dpi=300, bbox_inches="tight")
        plt.close()
        print(f" Saved: esg_overall_performance.png")

        print(f"\nAll visualizations completed successfully!")

    def export_results_to_excel(self):
        
        print("\n" + "=" * 80)
        print("EXPORTING RESULTS TO EXCEL")
        print("=" * 80)

        output_path = os.path.join(METHOD_EXCEL, 'esg_evaluation_results.xlsx')
        
        with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:

            summary_data= []
            for pillar in self.pillars.keys():
                for method in self.methods:
                    summary_data.append({
                        'Pillar': pillar,
                        'Method': method,
                        'Overall Score': self.results['summary'][pillar]['overall_scores'][method],
                        'Recommended': 'Yes' if method == 
                        self.results['summary'][pillar]['recommended_method'] else 'No'
                    })
            pd.DataFrame(summary_data).to_excel(writer, sheet_name='Summary', index=False)

            for pillar in self.pillars.keys():
                sheet_name = f'{pillar[0]}_Consistency'
                self.results['internal_consistency'][pillar]['correlation_matrix'].to_excel(
                    writer, sheet_name=sheet_name
                )
            
            robustness_data = []
            for pillar in self.pillars.keys():
                for method in self.methods:
                    rob = self.results['robustness'][pillar][method]
                    robustness_data.append({
                        'Pillar': pillar,
                        'Method': method,
                        'Outlier Percentage': rob['outlier_percentage'],
                        'Skewness': rob['skewness'],
                        'Kurtosis': rob['kurtosis'],
                        'Range': rob['range'],
                        'IQR': rob['iqr'],
                        'Robust CV': rob['robust_cv']
                    })
            pd.DataFrame(robustness_data).to_excel(writer, sheet_name='Robustness', index=False)

            stability_data = []
            for pillar in self.pillars.keys():
                stab = self.results['stability'][pillar]
                for method in self.methods:
                    stability_data.append({
                        'Pillar': pillar,
                        'Method': method,
                        'Avg Rank Displacement': stab['rank_displacements'][method],
                        'Top-10 Overlap (%)': stab['top10_overlaps'][method]
                    })
            pd.DataFrame(stability_data).to_excel(writer, sheet_name='Stability', index=False)

            interpret_data = []
            for pillar in self.pillars.keys():
                for method in self.methods:
                    interp = self.results['interpretability'][pillar][method]
                    interpret_data.append({
                        'Pillar': pillar,
                        'Method': method,
                        'Mean': interp['mean'],
                        'Median': interp['median'],
                        'Std Dev': interp['std'],
                        'Range Utilization (%)': interp['range_utilization'],
                        'Normalized Entropy': interp['normalized_entropy'],
                        'Gini Coefficient': interp['gini_coefficient']
                    })
            pd.DataFrame(interpret_data).to_excel(writer, sheet_name='Interpretability', index=False)

            for pillar_name, df in self.pillars.items():
                sheet_name = f'{pillar_name[0]}_Scores'
                df.to_excel(writer, sheet_name=sheet_name)
        
        print(f" Results exported successfully to {output_path}")

    def run_complete_evaluation(self):
            
        print("\n" + "=" * 80)
        print("ESG SCORE EVALUATION - COMPREHENSIVE ANALYSIS")
        print("=" * 80)
        print("\nEvaluating alternative ESG index specifications:")
        print(" . Equal-weighted aggregation")
        print(" . PCA-based weighting")
        print(" . Entropy-based weighting")
        print(" . Variance-based weighting")
        print("\n" + "=" * 80)

        self.evaluate_internal_consistency()
        self.evaluate_robustnesss_to_extremes()
        self.evaluate_ranking_stability()
        self.evaluate_interpretability()
        self.generate_comparative_summary()

        self.create_visualizations()
        self.export_results_to_excel()

        print("\n" + "=" * 80)
        print("EVALUATION COMPLETE")
        print("=" * 80)
        print("\nGenerated outputs:")
        print(" . esg_internal_consistency.png")
        print(" . esg_robustness.png")
        print(" . esg_stability.png")
        print(" . esg_interpretability.png")
        print(" . esg_comparative_summary.xlsx")

        return self.results

def evaluate_esg_scores(E_combined, S_combined, G_combined):
            
    evaluator = ESGScoreEvaluator(E_combined, S_combined, G_combined)
    results = evaluator.run_complete_evaluation()
    return evaluator

if __name__ == "__main__":
    print('ESG Score Evaluation Framework')
    print('=' * 80)
    print('\nTo use this framework, call:')
    print("evaluator = evaluate_esg_scores(E_combined, S_combined, G_combined)")
    print("\nWhere E_combined, S_combined, and G_combined are DataFrames with")
    print("columns: ['Equal', 'PCA', 'Entropy', 'Variance']")
                        

ESG Score Evaluation Framework

To use this framework, call:
evaluator = evaluate_esg_scores(E_combined, S_combined, G_combined)

Where E_combined, S_combined, and G_combined are DataFrames with
columns: ['Equal', 'PCA', 'Entropy', 'Variance']


In [49]:
evaluator = evaluate_esg_scores(E_combined, S_combined, G_combined)


ESG SCORE EVALUATION - COMPREHENSIVE ANALYSIS

Evaluating alternative ESG index specifications:
 . Equal-weighted aggregation
 . PCA-based weighting
 . Entropy-based weighting
 . Variance-based weighting

CRITERION 1: INTERNAL CONSISTENCY

Environmental Pillar:
------------------------------------------------------------
Average Pairwise Correlation: 0.9432
Variance Explained by PC1: 0.9585
Mean Coefficient of Variation: 0.1316

 Correlation Matrix:
           Equal     PCA  Entropy  Variance
Equal     1.0000  0.9740   0.8954    0.9892
PCA       0.9740  1.0000   0.9008    0.9795
Entropy   0.8954  0.9008   1.0000    0.9200
Variance  0.9892  0.9795   0.9200    1.0000

Social Pillar:
------------------------------------------------------------
Average Pairwise Correlation: 0.9880
Variance Explained by PC1: 0.9918
Mean Coefficient of Variation: 0.0714

 Correlation Matrix:
           Equal     PCA  Entropy  Variance
Equal     1.0000  0.9893   0.9868    0.9894
PCA       0.9893  1.0000   0.

In [50]:
print(vars(evaluator))

{'E_combined':                   Equal        PCA    Entropy   Variance
Economy Year                                            
Albania 2012  56.376223  57.076533  43.296033  55.821764
        2013  57.505312  57.548399  44.217845  57.067094
        2014  55.650678  56.854038  42.124381  54.680435
        2015  57.758684  58.087750  43.519637  57.089430
        2016  60.016433  58.970730  45.907208  59.440577
...                 ...        ...        ...        ...
Ukraine 2016  41.806863  35.800373  25.763095  37.709147
        2017  42.346504  36.746246  26.360701  38.227052
        2018  41.964009  36.191487  26.044382  37.717636
        2019  42.112984  37.027856  26.020881  37.365726
        2020  43.570590  38.808983  27.507118  39.299493

[405 rows x 4 columns], 'S_combined':                   Equal        PCA    Entropy   Variance
Economy Year                                            
Albania 2012  42.895256  39.940394  35.151878  38.209488
        2013  40.689710  37.545148

# <span style="color:#e0bda8">6. Score Addition to the E, S, and G Dataframes </span>

In [51]:
df_env_ml = df_env_score.copy()
df_env_ml['Score'] = E_score_variance
df_env_ml.to_csv(os.path.join(DATA_INDEX, "df_env_ml.csv"))

df_soc_ml = df_soc_score.copy()
df_soc_ml['Score'] = S_score_variance
df_soc_ml.to_csv(os.path.join(DATA_INDEX, "df_soc_ml.csv"))

df_gov_ml = df_gov_score.copy()
df_gov_ml['Score'] = G_score_variance
df_gov_ml.to_csv(os.path.join(DATA_INDEX, "df_gov_ml.csv"))